# Viral Clip Cutter (Qwen3-ASR)

Long video in → captioned 9:16 short clips out.

1. **Qwen3-ASR-1.7B** transcribes; **Qwen3-ForcedAligner** (de/en/…) or **MMS CTC aligner** (Arabic) gives word timestamps.
2. **Qwen3-8B** (local, 4-bit) picks the moments with the best hook + payoff; audio/text heuristics are blended in.
3. Cuts are snapped to silences between words, the speaker is tracked into a 9:16 frame and animated captions are burned in.

Runtime → *Change runtime type* → **T4 GPU** (or better), then run the cells top to bottom.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

## 1. Install

In [ ]:
!apt-get -qq install -y fonts-noto-core > /dev/null
!pip -q install "qwen-asr>=0.0.6" "silero-vad>=5.1" "transformers>=4.51.0" accelerate "bitsandbytes>=0.45" opencv-python-headless yt-dlp
!pip -q install "git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git"
!python -c "import qwen_asr, silero_vad, ctc_forced_aligner, cv2, bitsandbytes; print('ok')"

## 2. Get the code
Clones this repository (store a `GITHUB_TOKEN` Colab secret if the repo is private).
Alternatively upload a `clipper.zip` (zip of the `clipper/` folder) when prompted.

In [ ]:
import os, pathlib, shutil
REPO = "abokhaled1982/scraper"  #@param {type:"string"}
BRANCH = "asr"  #@param {type:"string"}
token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
shutil.rmtree("/content/repo", ignore_errors=True)
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
ok = os.system(f"git clone -q --depth 1 -b {BRANCH} {url} /content/repo") == 0 and pathlib.Path("/content/repo/clipper").exists()
if not ok:
    from google.colab import files
    print("Clone failed - upload clipper.zip instead")
    up = files.upload()
    os.makedirs("/content/repo", exist_ok=True)
    os.system(f"unzip -q -o '{next(iter(up))}' -d /content/repo")
os.chdir("/content/repo")
print(sorted(p.name for p in pathlib.Path("clipper").glob("*.py")))

## 3. Source video (upload or URL)

In [ ]:
VIDEO_URL = ""  #@param {type:"string"}
import glob
os.makedirs("/content/input", exist_ok=True)
if VIDEO_URL:
    !yt-dlp -q -f "bv*[height<=1080]+ba/b[height<=1080]" --merge-output-format mp4 -o "/content/input/%(id)s.%(ext)s" "$VIDEO_URL"
else:
    from google.colab import files
    for name, data in files.upload().items():
        pathlib.Path("/content/input", name).write_bytes(data)
VIDEO = max(glob.glob("/content/input/*"), key=os.path.getmtime)
print(VIDEO)

## 4. Settings + run

In [ ]:
LANG = "auto"  #@param ["auto", "ar", "de", "en"]
MAX_CLIPS = 5  #@param {type:"integer"}
MIN_SEC = 15  #@param {type:"number"}
MAX_SEC = 60  #@param {type:"number"}
LLM_MODEL = "Qwen/Qwen3-8B"  #@param ["Qwen/Qwen3-8B", "Qwen/Qwen3-4B", "Qwen/Qwen3-14B"]
USE_LLM = True  #@param {type:"boolean"}
REFRAME = "auto"  #@param ["auto", "face", "center", "fit", "none"]
CAPTIONS = "karaoke"  #@param ["karaoke", "phrase", "word", "none"]
HOOK_TITLE = True  #@param {type:"boolean"}
CONTEXT = ""  #@param {type:"string"}

args = ["run", VIDEO, "-o", "/content/clips", "--lang", LANG, "-n", str(MAX_CLIPS), "--min", str(MIN_SEC),
        "--max", str(MAX_SEC), "--llm-model", LLM_MODEL, "--reframe", REFRAME, "--captions", CAPTIONS]
args += ["--context", CONTEXT] if CONTEXT else []
args += [] if USE_LLM else ["--no-llm"]
args += ["--hook-title"] if HOOK_TITLE else []

from clipper.__main__ import main
main(args)

## 5. Review

In [ ]:
import json, base64
from IPython.display import HTML, display
work = max(pathlib.Path("/content/clips").iterdir(), key=os.path.getmtime)
for item in json.loads((work / "summary.json").read_text()):
    mp4 = work / "clips" / item["file"]
    b64 = base64.b64encode(mp4.read_bytes()).decode()
    display(HTML(f'''<div style="display:flex;gap:16px;margin:12px 0">
      <video width="270" controls src="data:video/mp4;base64,{b64}"></video>
      <div><b>{item["score"]:.0f}</b> · {item["start"]:.1f}-{item["end"]:.1f}s · {item["source"]}<br>
      <b>{item["title"]}</b><br><i>{item["hook"]}</i><br>{item["reason"]}<br><small>{item["text"][:400]}</small></div></div>'''))

To adjust a cut by hand: edit `start`/`end` in `moments.json` in the output folder and re-render
(the transcript and moment detection are cached):

In [ ]:
main(["render", VIDEO, "-o", "/content/clips", "--reframe", REFRAME, "--captions", CAPTIONS] + (["--hook-title"] if HOOK_TITLE else []))

In [ ]:
shutil.make_archive("/content/clips_export", "zip", work)
from google.colab import files
files.download("/content/clips_export.zip")